# NumPy : Numerical Python
NumPy is the foundation of scientific computing and machine learning in Python.
pandas , scikit-learn , PyTorch , TensorFlow , OpenCV , SciPy ... are all built on its ideas.

the core object is the **ndarray** (n-dimensional array) :

    - all elements have the SAME type (dtype)
    - stored in ONE contiguous block of memory
    - operations run in compiled C code on the WHOLE array (vectorization)

    1. creating arrays          6. broadcasting
    2. dtypes                   7. linear algebra
    3. indexing , views         8. random numbers
    4. reshaping , stacking     9. sorting , searching
    5. math and aggregation    10. save / load
                               11. applications in AI

In [1]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## 0. Why NumPy ? list vs ndarray

In [2]:
import sys, time
N = 1_000_000
py_list = list(range(N))
arr = np.arange(N)

print("list  : ~", round((sys.getsizeof(py_list) + N * sys.getsizeof(1)) / 1e6, 1), "MB  (pointers + int objects)")
print("array :  ", arr.nbytes / 1e6, "MB  (raw 8-byte integers)")

t = time.perf_counter(); s1 = sum(x * x for x in py_list); t_list = time.perf_counter() - t
t = time.perf_counter(); s2 = int((arr * arr).sum()); t_np = time.perf_counter() - t
print(s1 == s2, f"| list: {t_list * 1000:.1f} ms | numpy: {t_np * 1000:.1f} ms | x{t_list / t_np:.0f} faster")

list  : ~ 36.0 MB  (pointers + int objects)
array :   8.0 MB  (raw 8-byte integers)
True | list: 39.2 ms | numpy: 4.2 ms | x9 faster


In [3]:
# the same operator means different things
print([1, 2, 3] * 2)              # list : repetition
print(np.array([1, 2, 3]) * 2)    # array : element-wise math
print([1, 2] + [3, 4])            # list : concatenation
print(np.array([1, 2]) + np.array([3, 4]))

[1, 2, 3, 1, 2, 3]
[2 4 6]
[1, 2, 3, 4]
[4 6]


## 1. Creating arrays

In [4]:
a = np.array([1, 2, 3])                     # from a list
m = np.array([[1, 2, 3], [4, 5, 6]])        # from a list of lists ---> 2D
for name, x in [("a", a), ("m", m)]:
    print(f"{name}: shape={x.shape} ndim={x.ndim} size={x.size} dtype={x.dtype} itemsize={x.itemsize} nbytes={x.nbytes}")
print(m)

a: shape=(3,) ndim=1 size=3 dtype=int64 itemsize=8 nbytes=24
m: shape=(2, 3) ndim=2 size=6 dtype=int64 itemsize=8 nbytes=48
[[1 2 3]
 [4 5 6]]


In [5]:
print(np.zeros((2, 3)))                  # filled with 0.0
print(np.ones(4, dtype=int))             # filled with 1
print(np.full((2, 2), 7.5))              # filled with a value
print(np.eye(3))                         # identity matrix
print(np.zeros_like(m))                  # same shape and dtype as m

[[0. 0. 0.]
 [0. 0. 0.]]
[1 1 1 1]
[[7.5 7.5]
 [7.5 7.5]]
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
[[0 0 0]
 [0 0 0]]


In [6]:
print(np.arange(0, 10, 2))               # like range , but returns an array (stop not included)
print(np.arange(0, 1, 0.25))             # float steps are allowed
print(np.linspace(0, 1, 5))              # 5 numbers from 0 to 1 , BOTH ends included
print(np.logspace(-4, -1, 4))            # 1e-4 ... 1e-1 : learning-rate grids

[0 2 4 6 8]
[0.   0.25 0.5  0.75]
[0.   0.25 0.5  0.75 1.  ]
[0.0001 0.001  0.01   0.1   ]


In [7]:
# a list of rows with different lengths cannot become a normal 2D array
try:
    np.array([[1, 2, 3], [4, 5]])
except ValueError as e:
    print("ValueError:", str(e)[:70], "...")

ValueError: setting an array element with a sequence. The requested array has an i ...


## 2. dtypes : every element has the same type

In [8]:
print(np.array([1, 2, 3]).dtype)            # int64
print(np.array([1, 2.5]).dtype)             # float64 : int upcast to float
print(np.array([1, True]).dtype)            # int64   : bool upcast to int
print(np.array([1, "a"]).dtype)             # <U21    : EVERYTHING becomes a string !
print(np.array([1, 2], dtype=np.float32).dtype)

int64
float64
int64
<U21
float32


In [9]:
x = np.array([3.9, -3.9, 250.7])
print(x.astype(int))                        # truncates towards zero (new array)
print(x.astype(np.float32).nbytes, "vs", x.nbytes, "bytes")

img = np.array([250, 10], dtype=np.uint8)
with np.errstate(over="ignore"):
    print("uint8 overflow:", img + np.uint8(10))          # 260 ---> 4 , 20
print("safe          :", np.clip(img.astype(np.int16) + 10, 0, 255).astype(np.uint8))
print("float32 precision:", np.float32(16_777_217))       # 2**24 + 1 cannot be stored

[  3  -3 250]
12 vs 24 bytes
uint8 overflow: [ 4 20]
safe          : [255  20]
float32 precision: 1.6777216e+07


## 3. Indexing , slicing , views and copies

In [10]:
a = np.arange(10, 20)
print(a)
print(a[0], a[-1], a[2:5], a[::3], a[::-1])

[10 11 12 13 14 15 16 17 18 19]
10 19 [12 13 14] [10 13 16 19] [19 18 17 16 15 14 13 12 11 10]


In [11]:
m = np.arange(1, 13).reshape(3, 4)
print(m)
print("m[1, 2]   =", m[1, 2])        # row 1 , column 2  (better than m[1][2])
print("row 1     =", m[1])           # or m[1, :]
print("column 2  =", m[:, 2])
print("sub-block =\n", m[:2, 1:3])
print("last col  =", m[:, -1])

[[ 1  2  3  4]
 [ 5  6  7  8]
 [ 9 10 11 12]]
m[1, 2]   = 7
row 1     = [5 6 7 8]
column 2  = [ 3  7 11]
sub-block =
 [[2 3]
 [6 7]]
last col  = [ 4  8 12]


In [12]:
# IMPORTANT : a slice is a VIEW (shares memory) , not a copy
a = np.arange(6)
v = a[1:4]
v[0] = 100
print("a after changing the slice:", a)
print("shares memory:", np.shares_memory(a, v))

c = a[1:4].copy()                  # an independent copy
c[0] = -1
print("a after changing the copy :", a)

a after changing the slice: [  0 100   2   3   4   5]
shares memory: True
a after changing the copy : [  0 100   2   3   4   5]


In [13]:
# fancy indexing (a list of indices) ---> returns a COPY
a = np.arange(10, 20)
print(a[[0, 3, 7]])
m = np.arange(1, 13).reshape(3, 4)
print(m[[0, 2]])                    # rows 0 and 2
print(m[[0, 1, 2], [3, 2, 1]])      # pairs : (0,3) , (1,2) , (2,1)

[10 13 17]
[[ 1  2  3  4]
 [ 9 10 11 12]]
[ 4  7 10]


In [14]:
# boolean indexing : filter with a condition (mask)
scores = np.array([0.91, 0.42, 0.77, 0.15, 0.88])
mask = scores > 0.5
print(mask)
print(scores[mask])
print(scores[(scores > 0.3) & (scores < 0.8)])       # & | ~  with parentheses
scores[scores < 0.2] = 0.0                            # assign through a mask
print(scores)
print("how many > 0.5:", (scores > 0.5).sum(), "| any:", (scores > 0.95).any(), "| all:", (scores >= 0).all())

[ True False  True False  True]
[0.91 0.77 0.88]
[0.42 0.77]
[0.91 0.42 0.77 0.   0.88]
how many > 0.5: 3 | any: False | all: True


In [15]:
# np.where : vectorized if/else  ,  and indices where a condition is True
probs = np.array([0.2, 0.7, 0.55, 0.9, 0.1])
print(np.where(probs >= 0.5, 1, 0))
print(np.where(probs >= 0.5, "spam", "ham"))
print(np.flatnonzero(probs >= 0.5))                    # positions of the True values

[0 1 1 1 0]
['ham' 'spam' 'spam' 'spam' 'ham']
[1 2 3]


## 4. Reshaping and combining

In [16]:
a = np.arange(12)
print(a.reshape(3, 4))
print(a.reshape(2, -1).shape)            # -1 : 'compute this dimension for me'
print(a.reshape(3, 4).T)                 # transpose
try:
    a.reshape(5, 3)
except ValueError as e:
    print("ValueError:", e)

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
(2, 6)
[[ 0  4  8]
 [ 1  5  9]
 [ 2  6 10]
 [ 3  7 11]]
ValueError: cannot reshape array of size 12 into shape (5,3)


In [17]:
# add / remove an axis : very common with batches of data
x = np.array([1.0, 2.0, 3.0])                 # shape (3,)
print(x[np.newaxis, :].shape, x[:, np.newaxis].shape, np.expand_dims(x, 0).shape)
batch = np.zeros((1, 28, 28))
print(batch.squeeze().shape)                   # remove axes of size 1
print(np.arange(6).reshape(2, 3).ravel(), np.arange(6).reshape(2, 3).flatten())

(1, 3) (3, 1) (1, 3)
(28, 28)
[0 1 2 3 4 5] [0 1 2 3 4 5]


In [18]:
a = np.array([[1, 2], [3, 4]])
b = np.array([[5, 6], [7, 8]])
print(np.concatenate([a, b], axis=0))          # stack rows    (4, 2)
print(np.concatenate([a, b], axis=1))          # stack columns (2, 4)
print(np.stack([a, b]).shape)                  # NEW axis      (2, 2, 2)
print(np.vstack([a, b]).shape, np.hstack([a, b]).shape)
print([p.tolist() for p in np.split(np.arange(6), 3)])

[[1 2]
 [3 4]
 [5 6]
 [7 8]]
[[1 2 5 6]
 [3 4 7 8]]
(2, 2, 2)
(4, 2) (2, 4)
[[0, 1], [2, 3], [4, 5]]


## 5. Vectorized math (ufuncs) and aggregation

In [19]:
x = np.array([1.0, 4.0, 9.0])
print(x + 1, x * 2, x ** 2)
print(np.sqrt(x), np.log(x).round(3), np.exp([0, 1]).round(3))
print(np.maximum(x - 4, 0))                    # ReLU
print(np.round(np.sin(np.pi / 2), 3), np.abs([-2, 3]))

[ 2.  5. 10.] [ 2.  8. 18.] [ 1. 16. 81.]
[1. 2. 3.] [0.    1.386 2.197] [1.    2.718]
[0. 0. 5.]
1.0 [2 3]


In [20]:
# aggregation along an AXIS : axis=0 ---> down the rows (per column) , axis=1 ---> across columns (per row)
sales = np.array([[10, 20, 30],
                  [40, 50, 60]])               # 2 stores x 3 months
print("total      :", sales.sum())
print("per month  :", sales.sum(axis=0))       # collapse axis 0 (rows)
print("per store  :", sales.sum(axis=1))       # collapse axis 1 (columns)
print("mean/month :", sales.mean(axis=0))
print("best month index per store:", sales.argmax(axis=1))
print("keepdims   :", sales.sum(axis=1, keepdims=True).shape)
print("cumsum     :", np.cumsum([1, 2, 3, 4]))

total      : 210
per month  : [50 70 90]
per store  : [ 60 150]
mean/month : [25. 35. 45.]
best month index per store: [2 2]
keepdims   : (2, 1)
cumsum     : [ 1  3  6 10]


In [21]:
# missing values : np.nan
x = np.array([1.0, np.nan, 3.0])
print(x.mean(), np.nanmean(x))                 # nan spreads ; nanmean ignores it
print(np.isnan(x), np.isnan(x).sum())
print(np.nan == np.nan)                        # always False ---> use np.isnan
print(np.nan_to_num(x, nan=0.0))

nan 2.0
[False  True False] 1
False
[1. 0. 3.]


## 6. Broadcasting : operations between arrays of DIFFERENT shapes

compare the shapes from the RIGHT ; two dimensions are compatible if they are EQUAL or one of them is 1 :

    (3, 4)  +  (4,)     --->  (3, 4)      row vector added to every row
    (3, 4)  +  (3, 1)   --->  (3, 4)      column vector added to every column
    (3, 1)  +  (1, 4)   --->  (3, 4)      'outer' operation
    (3, 4)  +  (3,)     --->  ERROR

In [22]:
m = np.arange(12).reshape(3, 4)
print(m + 100)                                  # scalar
print(m + np.array([0, 10, 20, 30]))            # (4,)   ---> every row
print(m + np.array([[0], [100], [200]]))        # (3, 1) ---> every column
try:
    m + np.array([1, 2, 3])
except ValueError as e:
    print("ValueError:", e)

[[100 101 102 103]
 [104 105 106 107]
 [108 109 110 111]]
[[ 0 11 22 33]
 [ 4 15 26 37]
 [ 8 19 30 41]]
[[  0   1   2   3]
 [104 105 106 107]
 [208 209 210 211]]
ValueError: operands could not be broadcast together with shapes (3,4) (3,) 


In [23]:
# standardize every column (feature) at once
X = np.array([[20, 30_000], [30, 50_000], [40, 70_000]], dtype=float)
Z = (X - X.mean(axis=0)) / X.std(axis=0)        # (3,2) - (2,)  ---> broadcasting
print(Z.round(3))
print("mean ~ 0:", Z.mean(axis=0).round(10), "| std = 1:", Z.std(axis=0))

[[-1.225 -1.225]
 [ 0.     0.   ]
 [ 1.225  1.225]]
mean ~ 0: [0. 0.] | std = 1: [1. 1.]


In [24]:
# outer operations : all pairwise differences / distances without loops
a = np.array([1, 2, 3])
b = np.array([10, 20])
print(a[:, None] * b[None, :])                  # (3,1) * (1,2) ---> (3,2) multiplication table

P = np.array([[0, 0], [3, 4], [6, 8]], dtype=float)
D = np.sqrt(((P[:, None, :] - P[None, :, :]) ** 2).sum(axis=-1))   # (3,1,2)-(1,3,2) ---> (3,3,2)
print(D)

[[10 20]
 [20 40]
 [30 60]]
[[ 0.  5. 10.]
 [ 5.  0.  5.]
 [10.  5.  0.]]


## 7. Linear algebra

In [25]:
A = np.array([[2.0, 1.0], [1.0, 3.0]])
x = np.array([1.0, 2.0])
print("A @ x      =", A @ x)
print("A * A      =\n", A * A)                 # element-wise (Hadamard)
print("A @ A      =\n", A @ A)                 # matrix product
print("det        =", np.linalg.det(A).round(6))
print("inverse    =\n", np.linalg.inv(A).round(3))
print("A @ inv(A) =\n", (A @ np.linalg.inv(A)).round(10))
print("norm of x  =", np.linalg.norm(x).round(4))

A @ x      = [4. 7.]
A * A      =
 [[4. 1.]
 [1. 9.]]
A @ A      =
 [[ 5.  5.]
 [ 5. 10.]]
det        = 5.0
inverse    =
 [[ 0.6 -0.2]
 [-0.2  0.4]]
A @ inv(A) =
 [[ 1.  0.]
 [-0.  1.]]
norm of x  = 2.2361


In [26]:
# solve a linear system  A x = b   (better than inv(A) @ b : faster and more accurate)
#   2x +  y = 5
#    x + 3y = 10
b = np.array([5.0, 10.0])
print(np.linalg.solve(A, b))

[1. 3.]


In [27]:
# shape rules for @ : (n, k) @ (k, m) ---> (n, m)
X = np.ones((32, 784))          # a batch of 32 flattened 28x28 images
W = np.ones((784, 128))         # weights of a dense layer
print((X @ W).shape)
try:
    W @ X
except ValueError as e:
    print("ValueError:", str(e)[:60], "...")

(32, 128)
ValueError: matmul: Input operand 1 has a mismatch in its core dimension ...


## 8. Random numbers (reproducible)

In [28]:
rng = np.random.default_rng(seed=42)          # a Generator with its own seed
print(rng.random(3).round(3))                 # uniform [0, 1)
print(rng.integers(0, 10, size=5))            # integers [0, 10)
print(rng.normal(loc=0, scale=1, size=3).round(3))
print(rng.choice(["cat", "dog", "bird"], size=4, p=[0.5, 0.3, 0.2]))
print(rng.permutation(6))

rng2 = np.random.default_rng(seed=42)         # same seed ---> same numbers
print("reproducible:", np.allclose(rng2.random(3), np.random.default_rng(42).random(3)))

[0.774 0.439 0.859]
[0 6 2 0 5]
[ 0.128 -0.316 -0.017]
['cat' 'cat' 'bird' 'dog']
[1 0 4 2 3 5]
reproducible: True


In [29]:
# law of large numbers : the sample mean approaches the true mean
rng = np.random.default_rng(0)
for n in (10, 1_000, 100_000):
    sample = rng.normal(loc=170, scale=10, size=n)
    print(f"n={n:>7}: mean={sample.mean():.2f}  std={sample.std():.2f}")

n=     10: mean=170.85  std=7.41
n=   1000: mean=169.50  std=9.76
n= 100000: mean=170.00  std=10.00


## 9. Sorting , searching , counting

In [30]:
x = np.array([30, 10, 50, 20, 40])
print(np.sort(x), np.sort(x)[::-1])
order = np.argsort(x)                         # the INDICES that would sort x
print(order, x[order])

names = np.array(["e", "a", "c", "b", "d"])
print(names[np.argsort(-x)])                  # sort names by x , descending

[10 20 30 40 50] [50 40 30 20 10]
[1 3 0 4 2] [10 20 30 40 50]
['c' 'd' 'e' 'b' 'a']


In [31]:
# top-k without sorting everything : argpartition  O(n)
scores = np.random.default_rng(1).random(1_000_000)
k = 5
idx = np.argpartition(scores, -k)[-k:]        # the k largest (any order)
idx = idx[np.argsort(scores[idx])[::-1]]      # then sort only those k
print(idx, scores[idx].round(6))

[790141 323708 589332 996241 232216] [1.       0.999999 0.999998 0.999998 0.999997]


In [32]:
labels = np.array(["cat", "dog", "cat", "bird", "cat", "dog"])
values, counts = np.unique(labels, return_counts=True)
print(dict(zip(values.tolist(), counts.tolist())))
print(np.bincount([0, 2, 2, 1, 2]))                  # counts of 0 , 1 , 2 , ...
print(np.digitize([5, 15, 25], bins=[10, 20]))       # which bin : <10 , 10-20 , >=20
print(np.searchsorted([10, 20, 30], 25))             # insertion point in a sorted array

{'bird': 1, 'cat': 3, 'dog': 2}
[1 1 3]
[0 1 2]
2


## 10. Saving and loading arrays

In [33]:
from pathlib import Path
out = Path("numpy_demo"); out.mkdir(exist_ok=True)
X = np.arange(12, dtype=np.float32).reshape(3, 4)
y = np.array([0, 1, 0])

np.save(out / "X.npy", X)                            # one array , binary , keeps dtype and shape
np.savez_compressed(out / "data.npz", X=X, y=y)      # several arrays in one file
np.savetxt(out / "X.csv", X, delimiter=",", fmt="%.1f", header="a,b,c,d", comments="")

X2 = np.load(out / "X.npy")
with np.load(out / "data.npz") as d:
    y2 = d["y"]
X3 = np.loadtxt(out / "X.csv", delimiter=",", skiprows=1)
print(np.array_equal(X, X2), X2.dtype, y2, X3.dtype)

True float32 [0 1 0] float64


## 11. Applications in AI , Machine Learning and Expert Systems

In [34]:
# Application 1 : StandardScaler and KNN from the OOP session ---> rewritten with NumPy
class StandardScaler:
    def fit(self, X):
        X = np.asarray(X, dtype=float)
        self.mean_ = X.mean(axis=0)
        self.scale_ = X.std(axis=0)
        return self
    def transform(self, X):
        return (np.asarray(X, dtype=float) - self.mean_) / (self.scale_ + 1e-12)
    def fit_transform(self, X):
        return self.fit(X).transform(X)

class KNNClassifier:
    def __init__(self, k=5):
        self.k = k
    def fit(self, X, y):
        self.X_, self.y_ = np.asarray(X, float), np.asarray(y)
        return self
    def predict(self, X):
        X = np.asarray(X, float)
        d2 = ((X[:, None, :] - self.X_[None, :, :]) ** 2).sum(axis=-1)   # (n_test, n_train)
        nearest = np.argpartition(d2, self.k, axis=1)[:, : self.k]        # k closest per row
        votes = self.y_[nearest]                                          # (n_test, k)
        return (votes.mean(axis=1) > 0.5).astype(int)                     # majority (binary)
    def score(self, X, y):
        return (self.predict(X) == np.asarray(y)).mean()

rng = np.random.default_rng(1)
def make_circles(n):
    X = rng.uniform(-3, 3, size=(n, 2))
    y = (np.hypot(X[:, 0], X[:, 1]) > 2).astype(int)
    return X, y

X_tr, y_tr = make_circles(2000)
X_te, y_te = make_circles(1000)
scaler = StandardScaler().fit(X_tr)
t = time.perf_counter()
knn = KNNClassifier(k=5).fit(scaler.transform(X_tr), y_tr)
acc = knn.score(scaler.transform(X_te), y_te)
print(f"accuracy = {acc:.3f} | 1000 x 2000 distances in {(time.perf_counter() - t) * 1000:.0f} ms")

accuracy = 0.989 | 1000 x 2000 distances in 53 ms


In [35]:
# Application 2 : linear regression  ---> closed form (normal equation) and vectorized gradient descent
rng = np.random.default_rng(0)
n = 200
X = rng.uniform(0, 10, size=(n, 2))
true_w, true_b = np.array([3.0, -2.0]), 5.0
y = X @ true_w + true_b + rng.normal(0, 1.0, size=n)

Xb = np.hstack([X, np.ones((n, 1))])                  # add a column of 1 for the bias
w_closed = np.linalg.lstsq(Xb, y, rcond=None)[0]      # least squares
print("closed form :", w_closed.round(3))

mu, sd = X.mean(0), X.std(0)
Xs = (X - mu) / sd                                      # scale features for fast convergence
Xsb = np.hstack([Xs, np.ones((n, 1))])
w = np.zeros(3)
for epoch in range(500):
    grad = 2 / n * Xsb.T @ (Xsb @ w - y)               # ALL samples at once , no python loop over data
    w -= 0.1 * grad
w_orig = w[:2] / sd                                     # convert back to the ORIGINAL feature scale
b_orig = w[2] - (w[:2] * mu / sd).sum()
pred = Xsb @ w
r2 = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()
print("GD (scaled) :", w.round(3), " <--- weights for the SCALED features")
print("GD (orig)   :", np.append(w_orig, b_orig).round(3), "| R2 =", round(r2, 4))
print("true        :", np.append(true_w, true_b))

closed form : [ 3.034 -1.936  4.516]
GD (scaled) : [ 8.833 -5.766 11.032]  <--- weights for the SCALED features
GD (orig)   : [ 3.034 -1.936  4.516] | R2 = 0.9925
true        : [ 3. -2.  5.]


In [36]:
# Application 3 : softmax , cross-entropy , one-hot , accuracy , confusion matrix for a BATCH
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)              # stability , per row
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

logits = np.array([[2.0, 1.0, 0.1],
                   [0.5, 2.5, 0.2],
                   [1.2, 0.3, 3.0],
                   [2.2, 2.0, 0.1]])                  # 4 samples x 3 classes
y_true = np.array([0, 1, 2, 1])

P = softmax(logits)
one_hot = np.eye(3)[y_true]                           # fancy indexing on the identity matrix
loss = -np.mean(np.log(P[np.arange(len(y_true)), y_true] + 1e-12))
y_pred = P.argmax(axis=1)

cm = np.zeros((3, 3), dtype=int)
np.add.at(cm, (y_true, y_pred), 1)                    # count (true, pred) pairs
print(P.round(3))
print("one-hot:\n", one_hot)
print(f"loss={loss:.4f}  accuracy={(y_pred == y_true).mean():.2f}")
print("confusion matrix:\n", cm)

[[0.659 0.242 0.099]
 [0.11  0.809 0.081]
 [0.134 0.055 0.811]
 [0.515 0.422 0.063]]
one-hot:
 [[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]
 [0. 1. 0.]]
loss=0.4252  accuracy=0.75
confusion matrix:
 [[1 0 0]
 [1 1 0]
 [0 0 1]]


In [37]:
# Application 4 : a 2-layer neural network trained from scratch (forward + backpropagation)
rng = np.random.default_rng(0)
X_tr, y_tr = make_circles(400)
X_te, y_te = make_circles(200)
mu, sd = X_tr.mean(0), X_tr.std(0)
X_tr, X_te = (X_tr - mu) / sd, (X_te - mu) / sd

H = 16
W1, b1 = rng.normal(0, 1, (2, H)), np.zeros(H)
W2, b2 = rng.normal(0, 1, (H, 1)), np.zeros(1)
sigmoid = lambda z: 1 / (1 + np.exp(-z))

def forward(X):
    h = np.tanh(X @ W1 + b1)                    # hidden layer   (n, 16)
    p = sigmoid(h @ W2 + b2).ravel()            # output (prob)  (n,)
    return h, p

lr = 0.5
for epoch in range(1, 2001):
    h, p = forward(X_tr)
    loss = -np.mean(y_tr * np.log(p + 1e-12) + (1 - y_tr) * np.log(1 - p + 1e-12))
    dz2 = (p - y_tr)[:, None] / len(y_tr)       # gradient of BCE + sigmoid
    dW2, db2 = h.T @ dz2, dz2.sum(0)
    dh = dz2 @ W2.T * (1 - h ** 2)              # back through tanh
    dW1, db1 = X_tr.T @ dh, dh.sum(0)
    W1 -= lr * dW1; b1 -= lr * db1; W2 -= lr * dW2; b2 -= lr * db2
    if epoch in (1, 100, 500, 2000):
        print(f"epoch {epoch:4d}  loss={loss:.4f}  train_acc={((p > 0.5) == y_tr).mean():.3f}")

_, p_te = forward(X_te)
print("test accuracy:", ((p_te > 0.5) == y_te).mean())

epoch    1  loss=0.9355  train_acc=0.525
epoch  100  loss=0.2909  train_acc=0.943
epoch  500  loss=0.1119  train_acc=0.990
epoch 2000  loss=0.0538  train_acc=0.998
test accuracy: 0.985


In [38]:
# Application 5 : an image is just an array (H x W x channels , uint8)
H_, W_ = 6, 8
img = np.zeros((H_, W_), dtype=np.uint8)
img[2:4, 3:6] = 255                                    # a white rectangle on black
print(img)
flipped = img[:, ::-1]                                 # horizontal flip (data augmentation)
cropped = img[1:5, 2:7]                                # crop
normalized = img.astype(np.float32) / 255.0            # uint8 [0,255] ---> float [0,1]
edges = np.abs(np.diff(img.astype(np.int16), axis=1))  # simple horizontal edge detector
print("flipped row 2:", flipped[2])
print("crop shape   :", cropped.shape, "| normalized max:", normalized.max())
print("edges row 2  :", edges[2])

rgb = np.stack([img, img // 2, np.zeros_like(img)], axis=-1)       # (H, W, 3)
gray = (rgb @ np.array([0.299, 0.587, 0.114])).astype(np.uint8)     # luminance formula
print("rgb shape:", rgb.shape, "| gray[2, 3]:", gray[2, 3])

[[  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0 255 255 255   0   0]
 [  0   0   0 255 255 255   0   0]
 [  0   0   0   0   0   0   0   0]
 [  0   0   0   0   0   0   0   0]]
flipped row 2: [  0   0 255 255 255   0   0   0]
crop shape   : (4, 5) | normalized max: 1.0
edges row 2  : [  0   0 255   0   0 255   0]
rgb shape: (6, 8, 3) | gray[2, 3]: 150


In [39]:
# Application 6 : semantic search with cosine similarity (the retrieval step of RAG)
docs = ["python loops tutorial", "deep learning with pytorch", "baking bread at home",
        "neural network training tips", "sourdough starter recipe"]
# toy 4-d 'embeddings' : [programming , machine learning , cooking , bread]
E = np.array([[0.9, 0.2, 0.0, 0.0],
              [0.5, 0.9, 0.0, 0.0],
              [0.0, 0.0, 0.8, 0.7],
              [0.3, 0.95, 0.0, 0.0],
              [0.0, 0.0, 0.6, 0.9]])
query = np.array([0.2, 1.0, 0.0, 0.0])                       # "how to train a model"

E_n = E / np.linalg.norm(E, axis=1, keepdims=True)          # normalize every row
q_n = query / np.linalg.norm(query)
sims = E_n @ q_n                                             # ALL cosine similarities at once
for i in np.argsort(-sims)[:3]:
    print(f"{sims[i]:.3f}  {docs[i]}")

0.994  neural network training tips
0.952  deep learning with pytorch
0.404  python loops tutorial


In [40]:
# Application 7 : evaluate expert-system rules on 100,000 machines at once (vectorized masks)
rng = np.random.default_rng(3)
n = 100_000
temp = rng.normal(75, 8, n)
vibration = rng.gamma(2.0, 2.0, n)
pressure = rng.normal(3.0, 0.5, n)

rules = {
    "bearing failure": (temp > 85) & (vibration > 7.1),
    "cooling problem": (temp > 85) & ~(vibration > 7.1),
    "leak suspected": pressure < 2.0,
}
for name, mask in rules.items():
    print(f"{name:<16} {mask.sum():>6} machines ({mask.mean():.2%})")

any_alarm = np.logical_or.reduce(list(rules.values()))
print(f"{'any alarm':<16} {any_alarm.sum():>6} machines | first ids: {np.flatnonzero(any_alarm)[:5]}")

bearing failure    1388 machines (1.39%)
cooling problem    9255 machines (9.25%)
leak suspected     2296 machines (2.30%)
any alarm         12690 machines | first ids: [ 0  9 21 22 26]


In [41]:
# cleanup of the demo folder (set to True to delete it)
import shutil
CLEANUP = False
if CLEANUP:
    shutil.rmtree("numpy_demo")
print("numpy_demo kept:", Path("numpy_demo").exists())

numpy_demo kept: True


## Exercises
1. create a 5x5 matrix with 1 on the border and 0 inside (no loops).
2. given `x = np.arange(20)` , select the even numbers greater than 7 with a mask.
3. why does `a = np.arange(5); b = a[1:3]; b[:] = 0` change `a` ? how do you avoid it ?
4. with broadcasting , normalize every ROW of a matrix so that each row sums to 1.
5. write `min_max_scale(X)` that scales each column to [0, 1] (watch out for constant columns).
6. compute the accuracy and the 3x3 confusion matrix for `y_true = [0,1,2,2,1,0,2]` , `y_pred = [0,2,2,2,1,0,1]` with NumPy only.
7. in Application 4 , change the hidden size `H` to 2 and to 64 , and the learning rate to 5.0 : what happens ?
8. add a 6th document and a query about bread to Application 6 : is the ranking correct ?